# CodeReviewer.NLP — CO1: Text Pre-processing
## Semantic Commit / PR Consistency Auditor

This notebook demonstrates and evaluates the **Course Outcome 1 (CO1)** text pre-processing pipeline for Git commit messages and diffs.

### Pre-processing Objectives & Architecture:
1. **Normalization**: Unicode NFC normalization, lowercasing, stripping URLs, Git trailers (`Signed-off-by`, `Co-authored-by`), issue references (`#123`, `GH-45`), and whitespace collapsing.
2. **Identifier Splitting**: Deconstructing programming identifiers (`camelCase`, `PascalCase`, `snake_case`, `kebab-case`, digits, acronyms like `HTTPServer` → `http server`).
3. **Domain-Preserving Stop-word Removal**: Removing standard grammatical stopwords while **preserving software action verbs** (`add`, `fix`, `remove`, `update`, `delete`, `refactor`).
4. **Stemming vs Lemmatization**: Side-by-side comparison between Porter Stemmer and WordNet Lemmatizer (**verb-first** priority to handle past/participle forms).
5. **Verb Inflection Analysis**: Grammatical classification of the first word (`base`, `past`, `gerund`, `third_person`) to detect imperative mood adherence.
6. **Explainability Contract**: Every function produces an `(output, evidence: dict)` tuple.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Set project root and load preprocessor
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from code_reviewer.preprocessor import (
    normalize,
    split_identifiers,
    tokenize,
    remove_stopwords,
    stem,
    lemmatize,
    compare_stem_lemma,
    analyze_verb_inflection,
    preprocess_pipeline,
)

train_df = pd.read_csv(ROOT / "data" / "processed" / "train_classifier.csv")
print(f"Loaded {len(train_df):,} training commit messages from train_classifier.csv")

Loaded 5,912 training commit messages from train_classifier.csv


## 1. Normalization on Real Commit Messages

Commit messages often contain issue IDs, URLs, Git trailers, and noisy whitespace. Below, we demonstrate normalization on real messages from the dataset.

In [2]:
samples = [
    "Fixed bug in parseJsonPayload() to prevent crash on null #1234\n\nSigned-off-by: Dev <d@e.com>",
    "Update API documentation at https://api.docs.org/v2/endpoints (refs GH-890)",
    "Refactor  database   connection pool to handle  timeouts...  \n\nCo-authored-by: Bob <b@b.com>",
    "fix(core): sanitize input for UTF-8 encoding (Fixes #555)",
]

rows = []
for s in samples:
    norm, ev = normalize(s)
    rows.append({
        "Original": s.replace("\n", " ")[:60] + ("..." if len(s) > 60 else ""),
        "Normalized": norm,
        "Removed URLs": ev["removed_urls"],
        "Removed Trailers": ev["removed_trailers"],
        "Removed Issues": ev["removed_issues"],
    })

pd.DataFrame(rows)

,Original,Normalized,Removed URLs,Removed Trailers,Removed Issues
0,Fixed bug in parseJsonPayload() to prevent cra...,fixed bug in parsejsonpayload() to prevent cra...,[],[\nSigned-off-by: Dev <d@e.com>],[ #1234]
1,Update API documentation at https://api.docs.o...,update api documentation at ( ),[https://api.docs.org/v2/endpoints],[],[refs GH-890]
2,Refactor database connection pool to handle...,refactor database connection pool to handle ti...,[],[\nCo-authored-by: Bob <b@b.com>],[]
3,fix(core): sanitize input for UTF-8 encoding (...,fix(core): sanitize input for encoding ( ),[],[],"[ UTF-8, Fixes #555]"


## 2. Code Identifier Splitting

Software commits frequently reference technical identifiers. Classical NLP tokenizers treat `parseHTTPResponse` as an out-of-vocabulary token. Identifier splitting deconstructs camelCase, PascalCase, snake_case, kebab-case, acronyms, and embedded digits.

In [3]:
test_identifiers = [
    "getUserById",
    "HTTPServerResponse",
    "calculate_tax_rate",
    "user-service-client",
    "OAuth2Token",
    "utf8Encoding",
    "parseXMLDocument",
    "file123name",
]

id_results = []
for ident in test_identifiers:
    parts, ev = split_identifiers(ident)
    id_results.append({
        "Identifier": ident,
        "Split Tokens": " ".join(parts),
        "Token Count": len(parts),
        "Was Split": ev["was_split"],
    })

pd.DataFrame(id_results)

,Identifier,Split Tokens,Token Count,Was Split
0,getUserById,get user by id,4,True
1,HTTPServerResponse,http server response,3,True
2,calculate_tax_rate,calculate tax rate,3,True
3,user-service-client,user service client,3,True
4,OAuth2Token,oauth 2 token,3,True
5,utf8Encoding,utf 8 encoding,3,True
6,parseXMLDocument,parse xml document,3,True
7,file123name,file 123 name,3,True


## 3. Stop-word Removal with Action Verb Preservation

Standard NLTK stop-word lists drop action verbs and auxiliary words that carry semantic intent in commit messages. We evaluate token counts before and after stop-word removal across the 5,912 training commits.

In [4]:
import json
with open(ROOT / "results" / "phase2_preprocessing.json", "r") as f:
    phase2_res = json.load(f)

tc = phase2_res["token_counts"]
print(f"Total tokens before stop-word removal: {tc['total_before_stopwords']:,}")
print(f"Total tokens after stop-word removal:  {tc['total_after_stopwords']:,}")
print(f"Mean tokens per commit: {tc['mean_before_stopwords']} -> {tc['mean_after_stopwords']} (-{tc['reduction_percentage']}%)")

print("\nTop 10 removed stopwords:")
for word, cnt in tc["top_removed_stopwords"].items():
    print(f"  {word:10s}: {cnt:,}")

Total tokens before stop-word removal: 58,908
Total tokens after stop-word removal:  40,681
Mean tokens per commit: 9.96 -> 6.88 (-30.94%)

Top 10 removed stopwords:
  i         : 2,017
  to        : 1,781
  the       : 1,516
  in        : 1,117
  for       : 1,052
  is        : 684
  a         : 664
  of        : 627
  and       : 467
  on        : 451


## 4. Side-by-Side Comparison: Stemming vs. Lemmatization

A core CO1 requirement is evaluating **stemming (Porter)** against **lemmatization (WordNet verb-first)**.
- **Porter Stemmer**: Applies heuristic suffix chopping. Often creates non-words (e.g. `relational` → `relat`, `organization` → `organ`).
- **WordNet Lemmatizer (Verb-First)**: Resolves words to morphological roots using WordNet. Prioritizing verbs (`pos='v'`) ensures past-tense actions like `fixed` → `fix` and `added` → `add` are normalized to their base forms.

In [5]:
sample_words = [
    "fixed", "fixes", "fixing", "added", "adding", "updates", "updated",
    "updating", "refactored", "refactoring", "dependencies", "organization",
    "matrices", "indices", "relational", "capabilities", "authorization",
    "authentication", "configured", "configuring", "clean", "cleaned",
    "cleaning", "deleted", "deleting", "prevented", "preventing", "improved",
    "improving", "simplified", "simplifying"
]

comp_table = compare_stem_lemma(sample_words)
comp_rows = []
for word, d in comp_table.items():
    comp_rows.append({
        "Word": word,
        "Porter Stem": d["stem"],
        "WordNet Lemma (verb-first)": d["lemma"],
        "POS Used": d["pos_used"],
        "Different?": "Yes" if d["different"] else "No",
    })

pd.DataFrame(comp_rows)

,Word,Porter Stem,WordNet Lemma (verb-first),POS Used,Different?
0,added,ad,add,v,Yes
1,adding,ad,add,v,Yes
2,authentication,authent,authentication,none,Yes
3,authorization,author,authorization,none,Yes
4,capabilities,capabl,capability,n,Yes
5,clean,clean,clean,none,No
6,cleaned,clean,clean,v,No
7,cleaning,clean,clean,v,No
8,configured,configur,configure,v,Yes
9,configuring,configur,configure,v,Yes


### Vocabulary-Level Conflation Metrics & Over-Stemming Analysis

Beyond individual sample words, we evaluate stemming and lemmatization across the entire vocabulary (7,114 words) of the training dataset:
- **Differing Count**: 3,171 / 7,114 (44.57%) words produce differing stems vs lemmas.
- **Conflation Metric**: Stemming collapses surface forms more aggressively (5,283 distinct stems, avg 1.35 forms/stem) compared to lemmatization (5,745 distinct lemmas, avg 1.24 forms/lemma).
- **Semantic False-Merges (Over-Stemming)**: The heuristic suffix stripping of the Porter stemmer frequently conflates semantically unrelated words into identical stems.

In [6]:
import json
import pandas as pd

with open(ROOT / "results" / "phase2_preprocessing.json", "r", encoding="utf-8") as f:
    p2_data = json.load(f)

sl_meta = p2_data["stem_vs_lemma"]
print(f"Total vocabulary evaluated: {sl_meta['total_vocabulary_evaluated']:,}")
print(f"Words differing (stem != lemma): {sl_meta['differing_word_count']:,} ({sl_meta['differing_percentage']}%)")
print(f"Distinct stems:  {sl_meta['distinct_stems']:,} (avg {sl_meta['avg_surface_forms_per_stem']} surface forms/stem)")
print(f"Distinct lemmas: {sl_meta['distinct_lemmas']:,} (avg {sl_meta['avg_surface_forms_per_lemma']} surface forms/lemma)")

print("\n15 Verified Semantic False-Merges (Over-Stemming in Porter Stemmer):")
merges_df = pd.DataFrame(sl_meta["stemmer_semantic_false_merges_15"])
merges_df.columns = ["Porter Stem", "Conflated Words", "Semantic Conflict Explanation"]
merges_df

Total vocabulary evaluated: 7,114
Words differing (stem != lemma): 3,171 (44.57%)
Distinct stems:  5,283 (avg 1.35 surface forms/stem)
Distinct lemmas: 5,745 (avg 1.24 surface forms/lemma)

15 Verified Semantic False-Merges (Over-Stemming in Porter Stemmer):


,Porter Stem,Conflated Words,Semantic Conflict Explanation
0,organ,"[organization, organ]",Conflates administrative body/structure with b...
1,relat,"[relational, relate]",Conflates relational database theory with narr...
2,univers,"[universal, universe]",Conflates universal scope/selector with cosmol...
3,experi,"[experiment, experience]",Conflates empirical testing/experiment with pr...
4,gener,"[generate, general]",Conflates active synthesis/generation with gen...
5,anim,"[animation, animal]",Conflates UI motion animation with biological ...
6,digit,"[digital, digit]",Conflates electronic digital computing with si...
7,natur,"[natural, nature]",Conflates natural language / natural logarithm...
8,depart,"[department, depart]",Conflates organizational business unit with ve...
9,edit,"[edition, edit]",Conflates publication release version with tex...


## 5. Verb Inflection Analysis & Imperative Mood Adherence

Conventional commits mandate the **imperative mood** (e.g., `fix: add feature`, NOT `fix: added feature` or `fix: fixes feature`).
Here we classify the grammatical inflection of the first word across all 5,912 training commit messages:
- `base`: Imperative verb (e.g., `fix`, `add`, `update`)
- `past`: Past tense (e.g., `fixed`, `added`, `built`)
- `gerund`: Present participle (e.g., `fixing`, `adding`)
- `third_person`: 3rd person singular present (e.g., `fixes`, `adds`)
- `unknown`: Non-verb / non-standard first token

### Documented Ambiguity:
Verbs such as `set`, `put`, `cut`, `split`, `hit`, `read`, and `let` have identical base and past forms. In Git commit messages, these are defaulted to the imperative (`base`) mood and flagged as ambiguous in the evidence contract.

In [7]:
infl_info = phase2_res["inflection_analysis"]
print(f"Total messages evaluated: {infl_info['total_evaluated']:,}")
print(f"Imperative adherence (base form): {infl_info['imperative_adherence_percentage']}%")
print(f"Convention violations (past, gerund, 3rd person): {infl_info['convention_violations_percentage']}%")
print(f"Ambiguous homographs detected (set/put/cut/read): {infl_info['ambiguous_homographs_detected']}")

dist_rows = []
for infl_type, stats in infl_info["distribution"].items():
    dist_rows.append({
        "Inflection Type": infl_type,
        "Count": stats["count"],
        "Share (%)": f"{stats['percentage']:.2f}%",
        "Top Words": ", ".join(stats["top_words"].keys()),
    })

pd.DataFrame(dist_rows)

Total messages evaluated: 5,912
Imperative adherence (base form): 86.74%
Convention violations (past, gerund, 3rd person): 11.43%
Ambiguous homographs detected (set/put/cut/read): 89


,Inflection Type,Count,Share (%),Top Words
0,base,5128,86.74%,"fix, add, remove, use, update"
1,past,299,5.06%,"added, fixed, removed, updated, undefined"
2,gerund,163,2.76%,"missing, adding, fixing, removing, using"
3,third_person,214,3.62%,"always, fixes, adds, updates, tests"
4,unknown,108,1.83%,"broken, has, forgotten, done, misspelt"


## 6. End-to-End Pre-processing Pipeline with Evidence Contract

Finally, we run `preprocess_pipeline` with both `mode="lemma"` and `mode="stem"`, demonstrating the explainability evidence structure.

In [8]:
test_message = """Fixed critical bug in getUserById() to prevent nullPointer crash #432

Signed-off-by: Senior Dev <lead@company.com>"""

tokens_lemma, ev_lemma = preprocess_pipeline(test_message, mode="lemma")
tokens_stem, ev_stem = preprocess_pipeline(test_message, mode="stem")

print("=== RAW MESSAGE ===")
print(test_message)

print("\n=== LEMMA MODE OUTPUT ===")
print("Tokens:", tokens_lemma)
print("First Word Inflection:", ev_lemma["first_word_inflection"]["inflection"])
print("Removed Trailers:", ev_lemma["normalization"]["removed_trailers"])
print("Removed Issues:", ev_lemma["normalization"]["removed_issues"])

print("\n=== STEM MODE OUTPUT ===")
print("Tokens:", tokens_stem)


=== RAW MESSAGE ===
Fixed critical bug in getUserById() to prevent nullPointer crash #432

Signed-off-by: Senior Dev <lead@company.com>

=== LEMMA MODE OUTPUT ===
Tokens: ['fix', 'critical', 'bug', 'get', 'user', 'id', 'prevent', 'null', 'pointer', 'crash']
First Word Inflection: past
Removed Trailers: ['\nSigned-off-by: Senior Dev <lead@company.com>']
Removed Issues: [' #432']

=== STEM MODE OUTPUT ===
Tokens: ['fix', 'critic', 'bug', 'get', 'user', 'id', 'prevent', 'null', 'pointer', 'crash']
